# Aula 17 - Notebook: Problemas Hamiltonianos (TSP) e Roteamento do AGV de Amostragem

**Projeto:** SCADA-Core Automática — GRUPO 4
**Unidade Fabril:** Linha Automatizada de Seleção, Pesagem Dinâmica e Classificação Óptica de Grãos com Separação Pneumática Tripla
**Engenharia de Automação e Controle** — ECAA08

---

### 1. Objetivos do Notebook
Neste notebook implementamos o otimizador logístico do veículo autoguiado `AGV-01`, que parte do Laboratório Central de Qualidade (`LAB-CQ`), coleta amostras de arroz em 5 postos da planta e retorna à doca:

1. **Auditoria da matriz de distâncias:** simetria, diagonal nula e verificação da desigualdade triangular, com fechamento métrico por Floyd-Warshall;
2. **Heurística construtiva do Vizinho Mais Próximo (NN):** construção gulosa da rota em $O(n^2)$, com rastreamento passo a passo;
3. **Busca local 2-Opt:** refinamento da rota por inversão de segmentos até um ótimo local 2-ótimo;
4. **Certificação do ótimo:** força bruta ($\frac{(n-1)!}{2}$ ciclos) e programação dinâmica de Held-Karp $O(n^2 2^n)$;
5. **Cotas inferiores:** Árvore Geradora Mínima (Prim) e cota 1-Tree de Held-Karp;
6. **Atividades de investigação da Aula 17:** dependência do vértice inicial e bloqueio do corredor `VIS-401` $\leftrightarrow$ `SILO-701`;
7. **Integração SCADA:** cálculo das tags `ROT_DIST_TOTAL`, tempo de ciclo e consumo energético do `AGV-01`.

### 2. Funções de Formatação Tabular e Matricial em ASCII Puro

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII pura com suporte a infinitos."""
    larguras = [max(len(str(r)), 7) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else f"{v:.1f}" if isinstance(v, float) else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

### 3. Postos de Parada do AGV-01 e Matriz de Distâncias da Planta de Grãos
Os dados abaixo são os mesmos da Seção 3 do arquivo `17 - Problemas Hamiltonianos e Roteamento de AGV.md` (6 estações, distâncias em metros pelos corredores de tráfego de AGVs).

In [2]:
INF = float('inf')

estacoes = [
    {"Índice": 0, "Tag": "LAB-CQ",   "Posto": "Laboratório Central e Doca de Carga do AGV", "Amostra": "Partida/retorno e bancada NIR"},
    {"Índice": 1, "Tag": "FN-101",   "Posto": "Funil de Recepção",                          "Amostra": "Arroz in natura (impurezas e umidade do lote)"},
    {"Índice": 2, "Tag": "BAL-301",  "Posto": "Estação de Pesagem Contínua",                "Amostra": "Calibração dinâmica de WT-301 / FT-301"},
    {"Índice": 3, "Tag": "VIS-401",  "Posto": "Túnel Óptico de Visão Computacional",        "Amostra": "Grãos antes da zona de ejeção"},
    {"Índice": 4, "Tag": "SILO-701", "Posto": "Silo de Grãos Nobres (Cat. A)",              "Amostra": "Certificação de lote Tipo 1"},
    {"Índice": 5, "Tag": "SILO-702", "Posto": "Silo de Grãos Comerciais (Cat. B)",          "Amostra": "Produto secundário / reteste"},
]
locais = [e["Tag"] for e in estacoes]

dist_mat = [
    [  0.0,  45.0,  65.0,  80.0, 110.0,  75.0],
    [ 45.0,   0.0,  20.0,  35.0,  95.0,  80.0],
    [ 65.0,  20.0,   0.0,  18.0,  85.0,  70.0],
    [ 80.0,  35.0,  18.0,   0.0,  25.0,  40.0],
    [110.0,  95.0,  85.0,  25.0,   0.0,  35.0],
    [ 75.0,  80.0,  70.0,  40.0,  35.0,   0.0],
]

print("=" * 96)
print("          POSTOS DE AMOSTRAGEM DO AGV-01 (PLANTA DE CLASSIFICAÇÃO DE GRÃOS — GRUPO 4)")
print("=" * 96)
print(formatar_tabela(estacoes))
print("\nMatriz de Distâncias D [m]:")
print(formatar_matriz(dist_mat, locais, locais))

          POSTOS DE AMOSTRAGEM DO AGV-01 (PLANTA DE CLASSIFICAÇÃO DE GRÃOS — GRUPO 4)
Índice | Tag      | Posto                                      | Amostra                                      
-------+----------+--------------------------------------------+----------------------------------------------
0      | LAB-CQ   | Laboratório Central e Doca de Carga do AGV | Partida/retorno e bancada NIR                
1      | FN-101   | Funil de Recepção                          | Arroz in natura (impurezas e umidade do lote)
2      | BAL-301  | Estação de Pesagem Contínua                | Calibração dinâmica de WT-301 / FT-301       
3      | VIS-401  | Túnel Óptico de Visão Computacional        | Grãos antes da zona de ejeção                
4      | SILO-701 | Silo de Grãos Nobres (Cat. A)              | Certificação de lote Tipo 1                  
5      | SILO-702 | Silo de Grãos Comerciais (Cat. B)          | Produto secundário / reteste                 

Matriz de Distâncias D [m

### 4. Auditoria da Matriz: Simetria, Diagonal e Desigualdade Triangular
A teoria da Seção 1.2 do material da aula (2-Opt eliminando cruzamentos, atalhos do Christofides e a garantia de $1{,}5\times$) **pressupõe** a desigualdade triangular $w(u,v) \leq w(u,k) + w(k,v)$. Antes de otimizar, o SCADA deve auditar a matriz recebida do levantamento de campo.

Quando a desigualdade é violada, existe um trajeto pelos corredores que passa por outra estação (sem parar nela) e é mais curto que o valor tabelado. A **distância efetiva** do AGV é então o menor caminho entre os postos, obtido pelo **fechamento métrico** com Floyd-Warshall em $O(n^3)$.

In [3]:
def auditar_matriz(D, rotulos):
    n = len(D)
    simetrica = all(D[i][j] == D[j][i] for i in range(n) for j in range(n))
    diagonal_nula = all(D[i][i] == 0 for i in range(n))
    violacoes = []
    for i in range(n):
        for j in range(i + 1, n):
            melhor_k, melhor_desvio = None, D[i][j]
            for k in range(n):
                if k in (i, j):
                    continue
                if D[i][k] + D[k][j] < melhor_desvio:
                    melhor_k, melhor_desvio = k, D[i][k] + D[k][j]
            if melhor_k is not None:
                violacoes.append({
                    "Par (u, v)": f"{rotulos[i]} <-> {rotulos[j]}",
                    "D[u][v] tabelado": D[i][j],
                    "Melhor desvio via k": rotulos[melhor_k],
                    "D[u][k] + D[k][v]": melhor_desvio,
                    "Excesso (m)": D[i][j] - melhor_desvio,
                })
    return simetrica, diagonal_nula, violacoes

def fechamento_metrico(D):
    """Floyd-Warshall: menor distância real entre todos os pares de postos."""
    n = len(D)
    M = [linha[:] for linha in D]
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if M[i][k] + M[k][j] < M[i][j]:
                    M[i][j] = M[i][k] + M[k][j]
    return M

simetrica, diag_ok, violacoes = auditar_matriz(dist_mat, locais)
print("=" * 96)
print("                AUDITORIA DA MATRIZ DE DISTÂNCIAS DO AGV-01")
print("=" * 96)
print(f"Matriz simétrica (sTSP):         {simetrica}")
print(f"Diagonal nula:                   {diag_ok}")
print(f"Pares que violam a desig. triang.: {len(violacoes)} de {len(locais) * (len(locais) - 1) // 2}\n")
print(formatar_tabela(violacoes))

dist_efetiva = fechamento_metrico(dist_mat)
print("\nMatriz de Distâncias Efetivas (fechamento métrico por Floyd-Warshall) [m]:")
print(formatar_matriz(dist_efetiva, locais, locais))
_, _, violacoes_pos = auditar_matriz(dist_efetiva, locais)
print(f"\nViolações remanescentes após o fechamento métrico: {len(violacoes_pos)}")

                AUDITORIA DA MATRIZ DE DISTÂNCIAS DO AGV-01
Matriz simétrica (sTSP):         True
Diagonal nula:                   True
Pares que violam a desig. triang.: 5 de 15

Par (u, v)           | D[u][v] tabelado | Melhor desvio via k | D[u][k] + D[k][v] | Excesso (m)
---------------------+------------------+---------------------+-------------------+------------
LAB-CQ <-> SILO-701  | 110.0            | VIS-401             | 105.0             | 5.0        
FN-101 <-> SILO-701  | 95.0             | VIS-401             | 60.0              | 35.0       
FN-101 <-> SILO-702  | 80.0             | VIS-401             | 75.0              | 5.0        
BAL-301 <-> SILO-701 | 85.0             | VIS-401             | 43.0              | 42.0       
BAL-301 <-> SILO-702 | 70.0             | VIS-401             | 58.0              | 12.0       

Matriz de Distâncias Efetivas (fechamento métrico por Floyd-Warshall) [m]:
         |  LAB-CQ |  FN-101 | BAL-301 | VIS-401 | SILO-701 | SILO-702
-

**Leitura do resultado:** todas as violações passam por `VIS-401`, que fica no centro da malha de corredores. Por exemplo, ir de `FN-101` até `SILO-701` pela distância tabelada custa $95\text{ m}$, mas passando por `VIS-401` custa $35 + 25 = 60\text{ m}$. Nas próximas seções otimizamos a **matriz tabelada** (a mesma do material da aula) e, ao final, conferimos que a rota ótima não muda quando se usa a matriz efetiva.

### 5. Classe `RoteadorAGV_TSP`: Vizinho Mais Próximo, 2-Opt, Força Bruta, Held-Karp e Cotas Inferiores

In [4]:
from itertools import permutations
from typing import List, Tuple, Dict, Any

class RoteadorAGV_TSP:
    """Otimizador de rotas hamiltonianas do AGV de amostragem."""

    def __init__(self, locais: List[str], matriz_distancias: List[List[float]]):
        self.locais = locais
        self.dist = matriz_distancias
        self.n = len(locais)

    # ------------------------------------------------------------------ custo
    def calcular_custo_rota(self, rota: List[int]) -> float:
        """Custo do ciclo hamiltoniano fechado (inclui o retorno à base)."""
        c = sum(self.dist[rota[i]][rota[i + 1]] for i in range(len(rota) - 1))
        c += self.dist[rota[-1]][rota[0]]
        return c

    def nomes(self, rota: List[int]) -> str:
        return " -> ".join(self.locais[i] for i in rota) + f" -> {self.locais[rota[0]]}"

    # --------------------------------------------- heurística construtiva NN
    def vizinho_mais_proximo(self, inicio_idx: int = 0, registrar: bool = False):
        nao_vis = set(range(self.n))
        nao_vis.remove(inicio_idx)
        rota = [inicio_idx]
        atual = inicio_idx
        passos = []
        parcial = 0.0
        while nao_vis:
            # desempate determinístico pelo menor índice
            prox = min(nao_vis, key=lambda v: (self.dist[atual][v], v))
            parcial += self.dist[atual][prox]
            if registrar:
                passos.append({
                    "Passo": len(rota),
                    "Posto atual": self.locais[atual],
                    "Candidatos {posto: m}": ", ".join(f"{self.locais[v]}: {self.dist[atual][v]:.0f}" for v in sorted(nao_vis)),
                    "Escolhido": self.locais[prox],
                    "Custo parcial (m)": parcial,
                })
            rota.append(prox)
            nao_vis.remove(prox)
            atual = prox
        if registrar:
            passos.append({
                "Passo": len(rota),
                "Posto atual": self.locais[atual],
                "Candidatos {posto: m}": "retorno obrigatório à base",
                "Escolhido": self.locais[inicio_idx],
                "Custo parcial (m)": parcial + self.dist[atual][inicio_idx],
            })
            return rota, passos
        return rota

    # ------------------------------------------------------ busca local 2-Opt
    def otimizar_2opt(self, rota: List[int]) -> Tuple[List[int], List[Dict[str, Any]]]:
        """Aplica trocas 2-Opt (primeira melhoria) até convergir para um ótimo local."""
        melhor = list(rota)
        melhor_c = self.calcular_custo_rota(melhor)
        historico = []
        melhorou = True
        while melhorou:
            melhorou = False
            for i in range(1, self.n - 1):
                for j in range(i + 1, self.n):
                    nova = melhor[:i] + melhor[i:j + 1][::-1] + melhor[j + 1:]
                    c = self.calcular_custo_rota(nova)
                    if c < melhor_c:
                        historico.append({
                            "Troca": len(historico) + 1,
                            "Segmento invertido": " .. ".join([self.locais[melhor[i]], self.locais[melhor[j]]]),
                            "Custo antes (m)": melhor_c,
                            "Custo depois (m)": c,
                            "Delta (m)": c - melhor_c,
                        })
                        melhor_c = c
                        melhor = nova
                        melhorou = True
                        break
                if melhorou:
                    break
        return melhor, historico

    # ---------------------------------------------- métodos exatos (n pequeno)
    def forca_bruta(self, inicio_idx: int = 0) -> List[Tuple[float, List[int]]]:
        """Avalia as (n-1)! permutações com o início fixo, em O(n!)."""
        outros = [v for v in range(self.n) if v != inicio_idx]
        resultados = []
        for p in permutations(outros):
            rota = [inicio_idx] + list(p)
            resultados.append((self.calcular_custo_rota(rota), rota))
        resultados.sort(key=lambda x: (x[0], x[1]))
        return resultados

    def held_karp(self, inicio_idx: int = 0) -> Tuple[float, List[int]]:
        """Programação dinâmica de Held-Karp: O(n^2 * 2^n)."""
        outros = [v for v in range(self.n) if v != inicio_idx]
        custo = {}
        for k in outros:
            custo[(1 << k, k)] = (self.dist[inicio_idx][k], inicio_idx)
        # Subconjuntos S representados por máscaras de bits, em ordem crescente
        # (todo subconjunto de S é processado antes do próprio S).
        for mascara in range(1, 1 << self.n):
            if mascara & (1 << inicio_idx):
                continue
            for k in outros:
                if not (mascara & (1 << k)) or (mascara, k) in custo:
                    continue
                anterior = mascara & ~(1 << k)
                if anterior == 0:
                    continue
                candidatos = [(custo[(anterior, m)][0] + self.dist[m][k], m)
                              for m in outros if (anterior & (1 << m)) and (anterior, m) in custo]
                if candidatos:
                    custo[(mascara, k)] = min(candidatos)
        completa = sum(1 << k for k in outros)
        melhor_c, ultimo = min((custo[(completa, k)][0] + self.dist[k][inicio_idx], k) for k in outros)
        # reconstrução da rota
        rota = []
        mascara, k = completa, ultimo
        while k != inicio_idx:
            rota.append(k)
            _, pai = custo[(mascara, k)]
            mascara &= ~(1 << k)
            k = pai
        rota.append(inicio_idx)
        rota.reverse()
        return melhor_c, rota

    # --------------------------------------------------------- cotas inferiores
    def mst_prim(self, vertices: List[int] = None) -> Tuple[float, List[Tuple[int, int, float]]]:
        vertices = list(range(self.n)) if vertices is None else list(vertices)
        na_arvore = {vertices[0]}
        arestas = []
        total = 0.0
        while len(na_arvore) < len(vertices):
            w, u, v = min((self.dist[u][v], u, v) for u in na_arvore for v in vertices if v not in na_arvore)
            na_arvore.add(v)
            arestas.append((u, v, w))
            total += w
        return total, arestas

    def cota_1tree(self, excluido: int) -> float:
        """MST sobre V - {k} + as duas menores arestas incidentes em k."""
        restantes = [v for v in range(self.n) if v != excluido]
        mst, _ = self.mst_prim(restantes)
        duas_menores = sorted(self.dist[excluido][v] for v in restantes)[:2]
        return mst + sum(duas_menores)

agv_tsp = RoteadorAGV_TSP(locais, dist_mat)
print(f"Roteador instanciado com n = {agv_tsp.n} postos.")

Roteador instanciado com n = 6 postos.


### 6. Construção da Rota Inicial pela Heurística do Vizinho Mais Próximo (NN)
Rastreamento automático do exemplo resolvido da Seção 4.1 do material da aula.

In [5]:
r_nn, passos_nn = agv_tsp.vizinho_mais_proximo(0, registrar=True)
custo_nn = agv_tsp.calcular_custo_rota(r_nn)

print("=" * 110)
print("         RASTREAMENTO DA HEURÍSTICA DO VIZINHO MAIS PRÓXIMO (PARTIDA EM LAB-CQ)")
print("=" * 110)
print(formatar_tabela(passos_nn))
print(f"\nRota Inicial NN: {agv_tsp.nomes(r_nn)} | Distância: {custo_nn:.1f} m")

         RASTREAMENTO DA HEURÍSTICA DO VIZINHO MAIS PRÓXIMO (PARTIDA EM LAB-CQ)
Passo | Posto atual | Candidatos {posto: m}                                             | Escolhido | Custo parcial (m)
------+-------------+-------------------------------------------------------------------+-----------+------------------
1     | LAB-CQ      | FN-101: 45, BAL-301: 65, VIS-401: 80, SILO-701: 110, SILO-702: 75 | FN-101    | 45.0             
2     | FN-101      | BAL-301: 20, VIS-401: 35, SILO-701: 95, SILO-702: 80              | BAL-301   | 65.0             
3     | BAL-301     | VIS-401: 18, SILO-701: 85, SILO-702: 70                           | VIS-401   | 83.0             
4     | VIS-401     | SILO-701: 25, SILO-702: 40                                        | SILO-701  | 108.0            
5     | SILO-701    | SILO-702: 35                                                      | SILO-702  | 143.0            
6     | SILO-702    | retorno obrigatório à base                                

### 7. Refinamento por Busca Local 2-Opt
Aplicamos o 2-Opt em dois pontos de partida:
1. **Sobre a rota do NN:** se nenhuma troca reduzir o custo, a rota já é um **ótimo local 2-ótimo** (o 2-Opt funciona como certificado);
2. **Sobre a rota "ingênua"** citada na Seção 4.3 do material (`... -> VIS-401 -> SILO-702 -> SILO-701 -> LAB-CQ`, $268\text{ m}$): aqui o 2-Opt precisa trabalhar, e a tabela mostra a troca aplicada e o seu $\Delta$.

In [6]:
r_opt, hist_nn = agv_tsp.otimizar_2opt(r_nn)
print(">>> 2-Opt aplicado sobre a rota do Vizinho Mais Próximo")
print(f"Trocas de melhoria encontradas: {len(hist_nn)}")
print(f"Rota 2-Opt: {agv_tsp.nomes(r_opt)} | Distância: {agv_tsp.calcular_custo_rota(r_opt):.1f} m")
print("Diagnóstico: nenhuma inversão de segmento reduz o custo -> a rota NN já é 2-ótima.\n")

rota_ingenua = [0, 1, 2, 3, 5, 4]
custo_ingenua = agv_tsp.calcular_custo_rota(rota_ingenua)
r_corrigida, hist_ing = agv_tsp.otimizar_2opt(rota_ingenua)
custo_corrigida = agv_tsp.calcular_custo_rota(r_corrigida)

print(">>> 2-Opt aplicado sobre a rota ingênua (silos em ordem invertida)")
print(f"Rota ingênua:   {agv_tsp.nomes(rota_ingenua)} | Distância: {custo_ingenua:.1f} m")
print(formatar_tabela(hist_ing))
print(f"Rota corrigida: {agv_tsp.nomes(r_corrigida)} | Distância: {custo_corrigida:.1f} m")
print(f"Economia do 2-Opt: {custo_ingenua - custo_corrigida:.1f} m ({(custo_ingenua - custo_corrigida) / custo_ingenua * 100:.1f}%)")

# Verificação algébrica da troca: Delta = [w(vi,vj) + w(vi+1,vj+1)] - [w(vi,vi+1) + w(vj,vj+1)]
VIS, S701, S702, LAB = 3, 4, 5, 0
delta = (dist_mat[VIS][S701] + dist_mat[S702][LAB]) - (dist_mat[VIS][S702] + dist_mat[S701][LAB])
print(f"\nConferência pela fórmula do Delta: [D(VIS,S701) + D(S702,LAB)] - [D(VIS,S702) + D(S701,LAB)] = "
      f"[{dist_mat[VIS][S701]:.0f} + {dist_mat[S702][LAB]:.0f}] - [{dist_mat[VIS][S702]:.0f} + {dist_mat[S701][LAB]:.0f}] = {delta:.1f} m")

>>> 2-Opt aplicado sobre a rota do Vizinho Mais Próximo
Trocas de melhoria encontradas: 0
Rota 2-Opt: LAB-CQ -> FN-101 -> BAL-301 -> VIS-401 -> SILO-701 -> SILO-702 -> LAB-CQ | Distância: 218.0 m
Diagnóstico: nenhuma inversão de segmento reduz o custo -> a rota NN já é 2-ótima.

>>> 2-Opt aplicado sobre a rota ingênua (silos em ordem invertida)
Rota ingênua:   LAB-CQ -> FN-101 -> BAL-301 -> VIS-401 -> SILO-702 -> SILO-701 -> LAB-CQ | Distância: 268.0 m
Troca | Segmento invertido   | Custo antes (m) | Custo depois (m) | Delta (m)
------+----------------------+-----------------+------------------+----------
1     | SILO-702 .. SILO-701 | 268.0           | 218.0            | -50.0    
Rota corrigida: LAB-CQ -> FN-101 -> BAL-301 -> VIS-401 -> SILO-701 -> SILO-702 -> LAB-CQ | Distância: 218.0 m
Economia do 2-Opt: 50.0 m (18.7%)

Conferência pela fórmula do Delta: [D(VIS,S701) + D(S702,LAB)] - [D(VIS,S702) + D(S701,LAB)] = [25 + 75] - [40 + 110] = -50.0 m


### 8. Atividade 1 — Força Bruta e Held-Karp: Certificação do Ótimo Global
Fixando `LAB-CQ` como partida, há $(n-1)! = 5! = 120$ permutações dirigidas. Como a matriz é simétrica, cada ciclo aparece duas vezes (nos dois sentidos), restando $\frac{5!}{2} = 60$ ciclos hamiltonianos distintos.

In [7]:
resultados_bf = agv_tsp.forca_bruta(0)
ciclos_distintos = {min(tuple(r), tuple([r[0]] + r[1:][::-1])) for _, r in resultados_bf}
custo_otimo = resultados_bf[0][0]
n_otimas = sum(1 for c, _ in resultados_bf if c == custo_otimo)

print("=" * 90)
print("              CERTIFICAÇÃO DO ÓTIMO GLOBAL POR FORÇA BRUTA")
print("=" * 90)
print(f"Permutações dirigidas avaliadas: {len(resultados_bf)}")
print(f"Ciclos hamiltonianos distintos:  {len(ciclos_distintos)}")
print(f"Custos distintos observados:     {len(set(c for c, _ in resultados_bf))}")
print(f"Pior rota possível:              {resultados_bf[-1][0]:.1f} m\n")

top = []
vistos = set()
for c, r in resultados_bf:
    chave = min(tuple(r), tuple([r[0]] + r[1:][::-1]))
    if chave in vistos:
        continue
    vistos.add(chave)
    top.append({"Ranking": len(top) + 1, "Ciclo (um sentido)": agv_tsp.nomes(r), "Custo (m)": c})
    if len(top) == 5:
        break
print(formatar_tabela(top))
print(f"\nÓtimo global: {custo_otimo:.1f} m ({n_otimas} permutações dirigidas = 1 ciclo percorrido nos dois sentidos)")

custo_hk, rota_hk = agv_tsp.held_karp(0)
print(f"\nHeld-Karp (programação dinâmica): {agv_tsp.nomes(rota_hk)} | {custo_hk:.1f} m")
print(f"Gap da heurística NN + 2-Opt em relação ao ótimo: {(agv_tsp.calcular_custo_rota(r_opt) - custo_otimo) / custo_otimo * 100:.1f}%")

              CERTIFICAÇÃO DO ÓTIMO GLOBAL POR FORÇA BRUTA
Permutações dirigidas avaliadas: 120
Ciclos hamiltonianos distintos:  60
Custos distintos observados:     33
Pior rota possível:              460.0 m

Ranking | Ciclo (um sentido)                                                       | Custo (m)
--------+--------------------------------------------------------------------------+----------
1       | LAB-CQ -> FN-101 -> BAL-301 -> VIS-401 -> SILO-701 -> SILO-702 -> LAB-CQ | 218.0    
2       | LAB-CQ -> BAL-301 -> FN-101 -> VIS-401 -> SILO-701 -> SILO-702 -> LAB-CQ | 255.0    
3       | LAB-CQ -> FN-101 -> BAL-301 -> VIS-401 -> SILO-702 -> SILO-701 -> LAB-CQ | 268.0    
4       | LAB-CQ -> FN-101 -> SILO-702 -> SILO-701 -> VIS-401 -> BAL-301 -> LAB-CQ | 268.0    
5       | LAB-CQ -> FN-101 -> BAL-301 -> SILO-702 -> SILO-701 -> VIS-401 -> LAB-CQ | 275.0    

Ótimo global: 218.0 m (2 permutações dirigidas = 1 ciclo percorrido nos dois sentidos)

Held-Karp (programação dinâmica): LA

### 9. Atividade 2 — Dependência do Vértice Inicial
Executamos o NN a partir de cada posto (por exemplo, despacho de emergência a partir de `BAL-301`) e comparamos o custo do ciclo resultante.

In [8]:
tabela_inicio = []
for s in range(agv_tsp.n):
    r = agv_tsp.vizinho_mais_proximo(s)
    r2, h = agv_tsp.otimizar_2opt(r)
    tabela_inicio.append({
        "Partida": locais[s],
        "Rota NN": " -> ".join(locais[i] for i in r),
        "Custo NN (m)": agv_tsp.calcular_custo_rota(r),
        "Trocas 2-Opt": len(h),
        "Custo final (m)": agv_tsp.calcular_custo_rota(r2),
    })
print(formatar_tabela(tabela_inicio))
print("\nDiagnóstico: nesta instância, o NN chega ao ótimo de 218 m a partir de qualquer posto.")
print("Todas as rotas são rotações (ou inversões) do mesmo ciclo LAB-CQ, FN-101, BAL-301, VIS-401, SILO-701, SILO-702.")

Partida  | Rota NN                                                        | Custo NN (m) | Trocas 2-Opt | Custo final (m)
---------+----------------------------------------------------------------+--------------+--------------+----------------
LAB-CQ   | LAB-CQ -> FN-101 -> BAL-301 -> VIS-401 -> SILO-701 -> SILO-702 | 218.0        | 0            | 218.0          
FN-101   | FN-101 -> BAL-301 -> VIS-401 -> SILO-701 -> SILO-702 -> LAB-CQ | 218.0        | 0            | 218.0          
BAL-301  | BAL-301 -> VIS-401 -> SILO-701 -> SILO-702 -> LAB-CQ -> FN-101 | 218.0        | 0            | 218.0          
VIS-401  | VIS-401 -> BAL-301 -> FN-101 -> LAB-CQ -> SILO-702 -> SILO-701 | 218.0        | 0            | 218.0          
SILO-701 | SILO-701 -> VIS-401 -> BAL-301 -> FN-101 -> LAB-CQ -> SILO-702 | 218.0        | 0            | 218.0          
SILO-702 | SILO-702 -> SILO-701 -> VIS-401 -> BAL-301 -> FN-101 -> LAB-CQ | 218.0        | 0            | 218.0          

Diagnóstico: nesta inst

### 10. Atividade 3 — Cotas Inferiores: Árvore Geradora Mínima e 1-Tree
* **MST:** removendo uma aresta do ciclo ótimo obtém-se uma árvore geradora, logo $\text{Custo}(\text{MST}) \leq \text{Custo}(C^*)$.
* **1-Tree (Held-Karp):** para um vértice $k$, soma-se a MST de $V \setminus \{k\}$ às **duas** menores arestas incidentes em $k$ (no ciclo ótimo, $k$ tem exatamente duas arestas). A melhor cota é o máximo sobre todos os $k$.

In [9]:
custo_mst, arestas_mst = agv_tsp.mst_prim()
print("=" * 70)
print("        ÁRVORE GERADORA MÍNIMA (PRIM) DA MALHA DE POSTOS")
print("=" * 70)
print(formatar_tabela([{"Aresta": f"{locais[u]} -- {locais[v]}", "Peso (m)": w} for u, v, w in arestas_mst]))
print(f"\nCusto(MST) = {custo_mst:.1f} m  <=  Custo(TSP*) = {custo_otimo:.1f} m")

cotas = [{"Vértice excluído k": locais[k], "Cota 1-Tree (m)": agv_tsp.cota_1tree(k)} for k in range(agv_tsp.n)]
print("\n" + formatar_tabela(cotas))
melhor_cota = max(c["Cota 1-Tree (m)"] for c in cotas)
print(f"\nMelhor cota inferior 1-Tree: {melhor_cota:.1f} m")
print(f"Intervalo de certificação:   {melhor_cota:.1f} m <= TSP* <= {agv_tsp.calcular_custo_rota(r_opt):.1f} m")
print(f"Gap máximo garantido sem força bruta: {(agv_tsp.calcular_custo_rota(r_opt) - melhor_cota) / melhor_cota * 100:.2f}%")

        ÁRVORE GERADORA MÍNIMA (PRIM) DA MALHA DE POSTOS
Aresta               | Peso (m)
---------------------+---------
LAB-CQ -- FN-101     | 45.0    
FN-101 -- BAL-301    | 20.0    
BAL-301 -- VIS-401   | 18.0    
VIS-401 -- SILO-701  | 25.0    
SILO-701 -- SILO-702 | 35.0    

Custo(MST) = 143.0 m  <=  Custo(TSP*) = 218.0 m

Vértice excluído k | Cota 1-Tree (m)
-------------------+----------------
LAB-CQ             | 208.0          
FN-101             | 198.0          
BAL-301            | 178.0          
VIS-401            | 213.0          
SILO-701           | 183.0          
SILO-702           | 183.0          

Melhor cota inferior 1-Tree: 213.0 m
Intervalo de certificação:   213.0 m <= TSP* <= 218.0 m
Gap máximo garantido sem força bruta: 2.35%


### 11. Atividade 4 — Bloqueio do Corredor `VIS-401` $\leftrightarrow$ `SILO-701` por Empilhadeiras
Fazemos $D[3, 4] = D[4, 3] \leftarrow +\infty$ e recalculamos. Para que a rota continue fisicamente válida, o roteador nunca pode usar uma aresta de custo infinito.

In [10]:
dist_bloq = [linha[:] for linha in dist_mat]
dist_bloq[3][4] = dist_bloq[4][3] = INF
agv_bloq = RoteadorAGV_TSP(locais, dist_bloq)

r_nn_b = agv_bloq.vizinho_mais_proximo(0)
r_opt_b, hist_b = agv_bloq.otimizar_2opt(r_nn_b)
custo_b = agv_bloq.calcular_custo_rota(r_opt_b)
otimo_b = agv_bloq.forca_bruta(0)[0][0]

print("=" * 90)
print("     REROTEAMENTO DO AGV-01 COM O CORREDOR VIS-401 <-> SILO-701 INTERDITADO")
print("=" * 90)
print(f"Rota NN:        {agv_bloq.nomes(r_nn_b)} | {agv_bloq.calcular_custo_rota(r_nn_b):.1f} m")
print(f"Rota 2-Opt:     {agv_bloq.nomes(r_opt_b)} | {custo_b:.1f} m (trocas: {len(hist_b)})")
print(f"Ótimo (força bruta) com bloqueio: {otimo_b:.1f} m")
print(f"Impacto do bloqueio: +{custo_b - custo_otimo:.1f} m por ciclo (+{(custo_b - custo_otimo) / custo_otimo * 100:.1f}%)")
print("\nObservação: com o corredor interditado, a antiga rota 'ingênua' de 268 m passa a ser a ÓTIMA.")
print("O SILO-701 passa a ser visitado via SILO-702, e o retorno à doca sai do SILO-701 (110 m).")

     REROTEAMENTO DO AGV-01 COM O CORREDOR VIS-401 <-> SILO-701 INTERDITADO
Rota NN:        LAB-CQ -> FN-101 -> BAL-301 -> VIS-401 -> SILO-702 -> SILO-701 -> LAB-CQ | 268.0 m
Rota 2-Opt:     LAB-CQ -> FN-101 -> BAL-301 -> VIS-401 -> SILO-702 -> SILO-701 -> LAB-CQ | 268.0 m (trocas: 0)
Ótimo (força bruta) com bloqueio: 268.0 m
Impacto do bloqueio: +50.0 m por ciclo (+22.9%)

Observação: com o corredor interditado, a antiga rota 'ingênua' de 268 m passa a ser a ÓTIMA.
O SILO-701 passa a ser visitado via SILO-702, e o retorno à doca sai do SILO-701 (110 m).


### 12. Conferência com a Matriz de Distâncias Efetivas e Robustez do 2-Opt
Verificamos (a) se a rota ótima muda quando usamos a matriz corrigida pelo fechamento métrico e (b) de quantos pontos de partida aleatórios o 2-Opt consegue chegar ao ótimo global.

In [11]:
agv_efetivo = RoteadorAGV_TSP(locais, dist_efetiva)
custo_ef, rota_ef = agv_efetivo.held_karp(0)
print(f"Ótimo com a matriz efetiva (Held-Karp): {agv_efetivo.nomes(rota_ef)} | {custo_ef:.1f} m")
def ciclo_canonico(r):
    return min(tuple(r), tuple([r[0]] + r[1:][::-1]))

mesma_rota = ciclo_canonico(rota_ef) == ciclo_canonico(r_opt)
print(f"A rota ótima coincide com a da matriz tabelada? {mesma_rota}")

convergencia = {}
for _, rota_inicial in resultados_bf:
    final, _ = agv_tsp.otimizar_2opt(rota_inicial)
    c = agv_tsp.calcular_custo_rota(final)
    convergencia[c] = convergencia.get(c, 0) + 1
print("\nConvergência do 2-Opt partindo de cada uma das 120 permutações:")
print(formatar_tabela([{"Custo do ótimo local (m)": c, "Partidas que convergem": q,
                        "Percentual": f"{q / len(resultados_bf) * 100:.1f}%"} for c, q in sorted(convergencia.items())]))

Ótimo com a matriz efetiva (Held-Karp): LAB-CQ -> SILO-702 -> SILO-701 -> VIS-401 -> BAL-301 -> FN-101 -> LAB-CQ | 218.0 m
A rota ótima coincide com a da matriz tabelada? True

Convergência do 2-Opt partindo de cada uma das 120 permutações:
Custo do ótimo local (m) | Partidas que convergem | Percentual
-------------------------+------------------------+-----------
218.0                    | 120                    | 100.0%    


### 13. Integração com o SCADA-Core: Indicadores Operacionais do `AGV-01`
Premissas didáticas (as mesmas da Seção 4.3 do material): velocidade de cruzeiro $v = 1{,}0\text{ m/s}$ e consumo de $0{,}0005\text{ kWh/m}$. O número de ciclos por turno considera **somente o deslocamento**; o tempo de coleta em cada posto e o tempo de bancada devem ser somados quando forem medidos.

In [12]:
V_AGV = 1.0              # m/s
CONSUMO_KWH_M = 0.0005   # kWh por metro
TURNO_S = 8 * 3600

def indicadores(nome, custo):
    t = custo / V_AGV
    return {
        "Cenário": nome,
        "ROT_DIST_TOTAL (m)": f"{custo:.1f}",
        "Tempo de ciclo": f"{int(t // 60)} min {int(t % 60):02d} s",
        "Energia (kWh/ciclo)": f"{custo * CONSUMO_KWH_M:.3f}",
        "Ciclos/turno (só deslocamento)": int(TURNO_S // t),
    }

painel = [
    indicadores("Rota ingênua (silos invertidos)", custo_ingenua),
    indicadores("NN + 2-Opt (ótima)", agv_tsp.calcular_custo_rota(r_opt)),
    indicadores("Corredor VIS-401/SILO-701 bloqueado", custo_b),
]
print("=" * 110)
print("            PAINEL SCADA — MÓDULO INTRALOGÍSTICA / ROTEAMENTO AGV-01")
print("=" * 110)
print(formatar_tabela(painel))

tags_scada = {
    "ROT_SEQ_AGV": [locais[i] for i in r_opt] + [locais[r_opt[0]]],
    "ROT_DIST_TOTAL": agv_tsp.calcular_custo_rota(r_opt),
    "ROT_GAP_COTA_INF_%": round((agv_tsp.calcular_custo_rota(r_opt) - melhor_cota) / melhor_cota * 100, 2),
}
print("\nTags publicadas para o supervisório:")
for tag, valor in tags_scada.items():
    print(f"  {tag:<20} = {valor}")

            PAINEL SCADA — MÓDULO INTRALOGÍSTICA / ROTEAMENTO AGV-01
Cenário                             | ROT_DIST_TOTAL (m) | Tempo de ciclo | Energia (kWh/ciclo) | Ciclos/turno (só deslocamento)
------------------------------------+--------------------+----------------+---------------------+-------------------------------
Rota ingênua (silos invertidos)     | 268.0              | 4 min 28 s     | 0.134               | 107                           
NN + 2-Opt (ótima)                  | 218.0              | 3 min 38 s     | 0.109               | 132                           
Corredor VIS-401/SILO-701 bloqueado | 268.0              | 4 min 28 s     | 0.134               | 107                           

Tags publicadas para o supervisório:
  ROT_SEQ_AGV          = ['LAB-CQ', 'FN-101', 'BAL-301', 'VIS-401', 'SILO-701', 'SILO-702', 'LAB-CQ']
  ROT_DIST_TOTAL       = 218.0
  ROT_GAP_COTA_INF_%   = 2.35


### 14. Asserções e Validações Formais de Engenharia

In [13]:
# Validação 1: a rota é um ciclo hamiltoniano (cada posto exatamente uma vez, partindo da doca)
assert sorted(r_opt) == list(range(agv_tsp.n)), "Falha: a rota deve visitar todos os postos exatamente uma vez!"
assert r_opt[0] == 0, "Falha: a rota deve partir do LAB-CQ!"

# Validação 2: NN + 2-Opt atinge o ótimo global certificado por força bruta e por Held-Karp
assert agv_tsp.calcular_custo_rota(r_opt) == custo_otimo == custo_hk == 218.0, "Falha: o ótimo esperado é 218 m!"

# Validação 3: contagem combinatória (n - 1)!/2 = 60 ciclos distintos
assert len(resultados_bf) == 120 and len(ciclos_distintos) == 60, "Falha na contagem de ciclos hamiltonianos!"

# Validação 4: o 2-Opt corrige a rota ingênua (268 m -> 218 m) com Delta = -50 m
assert custo_ingenua == 268.0 and custo_corrigida == 218.0 and delta == -50.0, "Falha na correção 2-Opt!"

# Validação 5: as cotas inferiores são válidas (MST <= 1-Tree <= TSP*)
assert custo_mst <= melhor_cota <= custo_otimo, "Falha: cota inferior maior que o ótimo!"

# Validação 6: o bloqueio nunca é atravessado e eleva o custo para 268 m
assert custo_b == otimo_b == 268.0 and custo_b < INF, "Falha no reroteamento com corredor bloqueado!"

# Validação 7: o fechamento métrico elimina todas as violações e preserva o ótimo
assert len(violacoes) == 5 and len(violacoes_pos) == 0 and mesma_rota, "Falha na auditoria métrica!"

print("Todas as 7 asserções matemáticas e operacionais foram VALIDADAS com sucesso!")

Todas as 7 asserções matemáticas e operacionais foram VALIDADAS com sucesso!
